# 05 — Dashboard data prep

Builds Streamlit inputs under **`dashboard_data/`** by **reusing** outputs from notebook **03** (`airline_delay_summary.csv`, `airport_delay_summary.csv`, `seasonal_delay_summary.csv`) and **04** (`airport_clusters.csv`, `cluster_summary.csv`). Does **not** re-aggregate flight-level cause tables from `cleaned_flight_data.csv`.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".")
OUT = ROOT / "dashboard_data"
OUT.mkdir(parents=True, exist_ok=True)

CAUSES = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

## Airline dashboard slice ← `airline_delay_summary.csv` (notebook 03)

In [ ]:
src = ROOT / "airline_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create airline_delay_summary.csv")

a = pd.read_csv(src)
out = pd.DataFrame({
    "airline": a["Reporting_Airline"],
    "flights": a["flights"],
})
for c in CAUSES:
    out[f"total_{c}"] = a[f"total_{c}"]
    out[f"share_{c}"] = a[f"pct_{c}"] / 100.0

out["pct_delayed"] = np.nan
out["avg_arr_delay"] = np.nan

out.to_csv(OUT / "dashboard_airline_summary.csv", index=False)
out.head()

## Airport dashboard slice ← `airport_delay_summary.csv` + `airport_clusters.csv` (03 / 04)

In [ ]:
src = ROOT / "airport_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create airport_delay_summary.csv")

p = pd.read_csv(src)
o = pd.DataFrame({
    "airport": p["Origin"].astype(str),
    "role": "Origin",
    "flights": p["flights"],
    "pct_delayed": p["delay_rate"],
    "dominant_cause": p["dominant_cause"],
})
o["avg_arr_delay"] = np.nan
for c in CAUSES:
    o[f"total_{c}"] = p[f"total_{c}"]

ts = o[[f"total_{c}" for c in CAUSES]].sum(axis=1).replace(0, np.nan)
for c in CAUSES:
    o[f"share_{c}"] = (o[f"total_{c}"] / ts).fillna(0)

clus = ROOT / "airport_clusters.csv"
if clus.exists():
    cl = pd.read_csv(clus)[["Origin", "cluster"]].rename(columns={"Origin": "airport"})
    o = o.merge(cl, on="airport", how="left")
else:
    o["cluster"] = np.nan

o.to_csv(OUT / "dashboard_airport_summary.csv", index=False)
o.head()

## Monthly & seasonal dashboard slices ← `seasonal_delay_summary.csv` (notebook 03)

In [ ]:
src = ROOT / "seasonal_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create seasonal_delay_summary.csv")

s = pd.read_csv(src)

m = s[s["time_grain"] == "month"].copy()
m["Month"] = m["period"].astype(int)
month_out = pd.DataFrame({"Month": m["Month"]})
for c in CAUSES:
    month_out[f"total_{c}"] = m[f"total_{c}"].values
    month_out[f"avg_{c}"] = m[f"avg_{c}"].values
month_out["flights"] = np.nan
month_out["pct_delayed"] = np.nan
month_out["avg_arr_delay"] = np.nan
month_out.to_csv(OUT / "dashboard_monthly_summary.csv", index=False)

ss = s[s["time_grain"] == "season"].copy()
ss["Season"] = ss["period"].astype(str)
sea_out = pd.DataFrame({"Season": ss["Season"]})
for c in CAUSES:
    sea_out[f"total_{c}"] = ss[f"total_{c}"].values
    sea_out[f"avg_{c}"] = ss[f"avg_{c}"].values
sea_out["flights"] = np.nan
sea_out["pct_delayed"] = np.nan
sea_out["avg_arr_delay"] = np.nan

order = ["Winter", "Spring", "Summer", "Autumn"]
sea_out["Season"] = pd.Categorical(sea_out["Season"], categories=order, ordered=True)
sea_out = sea_out.sort_values("Season")
sea_out.to_csv(OUT / "dashboard_seasonal_summary.csv", index=False)

month_out.head(), sea_out

## Cluster files ← `cluster_summary.csv` (notebook 04); copy scatter file

In [ ]:
cs = ROOT / "cluster_summary.csv"
if cs.exists():
    df = pd.read_csv(cs)
    df.to_csv(OUT / "dashboard_cluster_summary.csv", index=False)

    cent_cols = [c for c in df.columns if c.startswith("centroid_")]
    long_rows = []
    for _, row in df.iterrows():
        for cc in cent_cols:
            long_rows.append({
                "cluster": int(row["cluster"]),
                "metric": cc.replace("centroid_", ""),
                "value": float(row[cc]),
            })
    pd.DataFrame(long_rows).to_csv(OUT / "dashboard_cluster_centroids.csv", index=False)
else:
    pd.DataFrame(columns=["cluster", "n_airports", "dominant_delay_type", "sample_airports"]).to_csv(
        OUT / "dashboard_cluster_summary.csv", index=False
    )
    pd.DataFrame(columns=["cluster", "metric", "value"]).to_csv(
        OUT / "dashboard_cluster_centroids.csv", index=False
    )

ac = ROOT / "airport_clusters.csv"
if ac.exists():
    shutil.copy2(ac, OUT / "airport_clusters.csv")

sorted(p.name for p in OUT.iterdir())